In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    silhouette_score
)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# 1. Dataset Setup

### Re-introduction of Dataset

- **Data**: Diabetes 130-US Hospitals for Years 1999–2008
- **Size**: 101,766 encounters × 50 features
- **Content**: Patient demographics, admission/discharge details, clinical measurements, diagnoses (ICD-9), lab results, medications, and readmission outcome
- **`IDS_mapping.csv`**: Mapping table for `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`

### Machine Learning Tasks
1. **Classification**: Predict whether insulin was used during the encounter (binary)
2. **Regression**: Predict length of hospital stay (`time_in_hospital`)
3. **Clustering**: Discover patient subgroups based on clinical and demographic features

### Changes Since Interim Submission
- **Outlier handling expanded**: The interim only applied IQR-based outlier removal to `num_lab_procedures`. Now we review all numeric fields and apply appropriate handling per variable.
- **Justifications strengthened**: Each drop, recode, and imputation decision now includes explicit clinical or statistical reasoning.
- **Outlier removal applied to training set only**: Following best practice, outliers are removed after the train/test split, only from the training data.
- **Diagnoses 2 and 3 added**: Originally only `diag_1` (primary diagnosis) was used. Now `diag_2` and `diag_3` (secondary diagnoses) are also one-hot encoded and included as features. The model learns separate coefficients per diagnosis position, naturally weighting primary higher than secondary based on the data. Patients without a secondary diagnosis fall into the 'Other' baseline category (all-zero rows).
- **OHE boolean fix**: Discovered that `pd.get_dummies()` creates boolean columns that get silently dropped by `select_dtypes(include=[np.number])`. Added explicit conversion to `int8` so all 44 features make it to the model.
- **Models improved based on experimentation**: For regression, after testing Linear, Ridge, ElasticNet, Decision Tree, and Neural Network, Ridge was replaced with **Neural Network (MLP)** — the only model with a meaningfully better R². Linear and Ridge produced identical results since multicollinearity was not an issue.
- **Log transformation tested but not applied**: We tried `log1p` transformation on `time_in_hospital` to address its right-skewed distribution. Skewness improved (1.18 → 0.13), but R² actually dropped (0.092 → 0.047) due to error amplification during inverse transform. Original scale was kept.
- **Full evaluation suite**: Confusion matrix, ROC curve, Precision/Recall/F1, R², silhouette score, PCA visualization.

### Notes from Feedback
- Classification (insulin): Features are restricted to information available at admission (demographics, clinical history, diagnoses, A1Cresult, admission_type)
- A1Cresult is the average blood sugar during 2–3 months before admission, so it is pre-admission information
- Outlier handling was limited to one variable — **now expanded to all relevant numeric fields** with clear justification for each decision
- Dropped or recoded values now include explicit reasoning

# 2. Data Cleaning & Preprocessing

## 2-0. Load Data

In [ ]:
# Load main dataset — '?' is the missing value marker in this dataset
df = pd.read_csv('Diabetic_Data.csv', na_values='?', low_memory=False)
print(f"Original shape: {df.shape}")
df.head(3)

In [ ]:
# Load IDS mapping tables
# The CSV contains three tables separated by header rows.
# We parse each table individually using skiprows and nrows.
import pandas as pd

adm_type = pd.read_csv('IDS_mapping.csv', skiprows=1, nrows=8, header=None, names=['id', 'description'])
discharge = pd.read_csv('IDS_mapping.csv', skiprows=11, nrows=29, header=None, names=['id', 'description'])
adm_source = pd.read_csv('IDS_mapping.csv', skiprows=42, nrows=26, header=None, names=['id', 'description'])

admission_type_map = dict(zip(adm_type['id'], adm_type['description']))
discharge_disposition_map = dict(zip(discharge['id'], discharge['description']))
admission_source_map = dict(zip(adm_source['id'], adm_source['description']))

print("Admission type mapping:", admission_type_map)

## 2-1. Drop Columns and Deduplicate

**Rationale for each decision:**
- **Keep only first encounter per patient**: Multiple encounters from the same patient violate independence assumptions for modeling. We keep the chronologically first encounter (sorted by `encounter_id`).
- **Drop `weight`**: Over 97% of values are missing — imputation would be unreliable and misleading.
- **Drop `payer_code`**: Insurance type has ~40% missing and is not clinically relevant to our prediction tasks (insulin use, length of stay).
- **Drop `encounter_id`, `patient_nbr`**: These are unique identifiers with no predictive value. Keeping them risks data leakage (models memorizing patient IDs).

In [ ]:
# Deduplicate: keep only the first encounter per patient
df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset=['patient_nbr'], keep='first')
print(f"After deduplication: {df.shape[0]} rows (one per patient)")

# Drop columns with justification
df = df.drop(columns=['weight', 'payer_code', 'encounter_id', 'patient_nbr'])
print(f"After column drop: {df.shape}")

## 2-2. Remove Expired and Hospice Discharges

Following Strack et al. (2014), we exclude encounters where the patient died or was transferred to hospice. These outcomes have fundamentally different clinical dynamics — predicting insulin use or length of stay is not meaningful for these cases.

- **11**: Expired
- **13**: Hospice / home
- **14**: Hospice / medical facility
- **19, 20, 21**: Expired variants (Medicaid hospice)

In [ ]:
exclude = [11, 13, 14, 19, 20, 21]
n_before = len(df)
df = df[~df['discharge_disposition_id'].isin(exclude)]
print(f"Removed {n_before - len(df)} expired/hospice encounters → {len(df)} remaining")

## 2-3. Handle Missing Values

**Rationale for each imputation:**
- **`race`** (2.2% missing): Filled with `'Unknown'`. Race is a low-cardinality categorical; dropping rows would lose data unnecessarily, and the missingness itself may carry information (e.g., recording practices).
- **`medical_specialty`** (49% missing): Filled with `'Missing'`. Nearly half the values are absent, suggesting many admissions did not record the attending specialty. This is informative — treating it as its own category preserves the signal.
- **`diag_1/2/3`** (<1% missing each): Filled with `'Unknown'`. Very small proportion; dropping would also work but filling preserves more data.
- **`A1Cresult`, `max_glu_serum`**: NaN means the test was **not ordered**, which is clinically different from a test with a normal result. We fill with `'None'` to represent "test not taken" as a distinct category.
- **`gender`**: 3 rows with `'Unknown/Invalid'` — removed because gender is a key demographic feature and 3 rows is negligible.

In [ ]:
# Impute missing values
df['race'] = df['race'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Missing')

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].fillna('Unknown')

# A1Cresult and max_glu_serum: NaN = test not taken (distinct from normal result)
df['A1Cresult'] = df['A1Cresult'].fillna('None')
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')

# Note on medical_specialty: 47% missing. Strack et al. (2014) kept this column,
# treating 'Missing' as a category. We fill it the same way for completeness,
# but DO NOT include it in our model features. Rationale:
#   1. With 73 unique specialties, OHE would explode the feature space.
#   2. Specialty assignment may partially reflect clinical judgments made
#      shortly after admission, blurring the leakage boundary.
# We exclude it to keep the feature set clean and pre-admission-only.

# Remove the 3 rows with invalid gender
n_before = len(df)
df = df[df['gender'] != 'Unknown/Invalid']
print(f"Removed {n_before - len(df)} rows with invalid gender")
print(f"Remaining missing values:\n{df.isna().sum()[df.isna().sum() > 0]}")

## 2-4. Outlier Review Across All Numeric Fields

The interim submission only checked `num_lab_procedures` for outliers. Now we systematically review **all continuous numeric variables** using the IQR method, and justify our handling decision for each.

**Important**: Actual outlier removal will be done **after the train/test split**, applied to the training set only. Here we analyze the full dataset to understand the distributions and decide our strategy.

In [ ]:
# Comprehensive outlier analysis for all continuous numeric variables
continuous_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

outlier_report = []
for col in continuous_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_out / len(df) * 100
    outlier_report.append({
        'Variable': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'Lower': lower, 'Upper': upper,
        'N_Outliers': n_out, 'Pct': round(pct, 2),
        'Max': df[col].max(), 'Skewness': round(df[col].skew(), 2)
    })

outlier_df = pd.DataFrame(outlier_report)
print(outlier_df.to_string(index=False))

In [ ]:
# Visualize distributions of key numeric variables
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), continuous_cols):
    ax.hist(df[col], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=10)
    ax.axvline(df[col].median(), color='red', linestyle='--', linewidth=1, label='Median')
    ax.legend(fontsize=8)
plt.suptitle('Distribution of Continuous Numeric Variables', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Outlier Handling Strategy — Variable-by-Variable Justification

| Variable | IQR Outliers | Decision | Reasoning |
|---|---|---|---|
| `num_lab_procedures` | 0.14% | **IQR cap** | Near-symmetric distribution; extreme values (>96) are rare data entry artifacts |
| `num_procedures` | 5.5% | **IQR cap** | Right-skewed; max=77 is clinically implausible (likely coding error). Cap at upper bound |
| `num_medications` | 2.7% | **IQR cap** | Right-skewed; values >35 are extreme polypharmacy. Cap to reduce influence |
| `number_diagnoses` | 0.34% | **No action** | Range 1–16 is clinically reasonable; all values are plausible |
| `time_in_hospital` | 2.0% | **No action** | This is our regression target. Removing outliers from the target would bias the model. Range 1–14 days is clinically valid |
| `number_outpatient` | 13.0% | **Percentile cap (99th)** | IQR=0 makes IQR method inapplicable. Most patients have 0 visits; values up to 42 represent heavy utilizers. Cap at 99th percentile to limit extreme influence |
| `number_emergency` | 7.3% | **Percentile cap (99th)** | Same reasoning as outpatient — zero-inflated distribution |
| `number_inpatient` | 11.7% | **Percentile cap (99th)** | Same reasoning — zero-inflated distribution |

**Note**: Capping (Winsorizing) is preferred over removal for features because:
1. We preserve the information that the patient had a high value (just bounded)
2. We don't lose rows (important for a dataset where rows represent unique patients)
3. Outlier removal is only appropriate for clearly erroneous values

Actual capping will be applied **only to training data** after the split.

## 2-5. ICD-9 Diagnosis Code Grouping

Based on Strack et al. (2014), Table 2. Maps raw ICD-9 codes to major clinical categories.

In [ ]:
def map_icd9_to_group(code):
    """Map an ICD-9 code string to a major clinical category."""
    if pd.isna(code) or code == 'Unknown':
        return 'Other'
    code_str = str(code).strip()
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    try:
        c = float(code_str)
    except ValueError:
        return 'Other'

    if 250 <= c < 251:      return 'Diabetes'
    if 390 <= c <= 459 or c == 785: return 'Circulatory'
    if 460 <= c <= 519 or c == 786: return 'Respiratory'
    if 520 <= c <= 579 or c == 787: return 'Digestive'
    if 800 <= c <= 999:     return 'Injury'
    if 710 <= c <= 739:     return 'Musculoskeletal'
    if 580 <= c <= 629 or c == 788: return 'Genitourinary'
    if 140 <= c <= 239:     return 'Neoplasms'
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_group'] = df[col].apply(map_icd9_to_group)

print('Primary diagnosis group distribution:')
print(df['diag_1_group'].value_counts())

## 2-6. Map IDS to Descriptive Labels

In [ ]:
df['admission_type']       = df['admission_type_id'].map(admission_type_map)
df['discharge_disposition'] = df['discharge_disposition_id'].map(discharge_disposition_map)
df['admission_source']     = df['admission_source_id'].map(admission_source_map)

for col in ['admission_type', 'discharge_disposition', 'admission_source']:
    df[col] = df[col].fillna('Other')

df = df.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'])

## 2-7. Age Group Labeling

In [ ]:
age_group_map = {
    '[0-10)': 'Young', '[10-20)': 'Young', '[20-30)': 'Young',
    '[30-40)': 'Middle', '[40-50)': 'Middle', '[50-60)': 'Middle',
    '[60-70)': 'Older', '[70-80)': 'Older', '[80-90)': 'Older', '[90-100)': 'Older'
}
df['age_group'] = df['age'].map(age_group_map)
print(df['age_group'].value_counts())

## 2-8. Target Variable Preparation

In [ ]:
# Classification target: binarize insulin usage
df['insulin_binary'] = (df['insulin'] != 'No').astype(int)
print('insulin_binary distribution:')
print(df['insulin_binary'].value_counts())
print(f'Class balance: {df["insulin_binary"].mean():.2%} insulin users')

# Readmission indicator (for clustering analysis)
df['readmit_30'] = (df['readmitted'] == '<30').astype(int)

# 3. Exploratory Data Analysis (EDA)

## 3-1. 30-Day Readmission Rate by Primary Diagnosis Group

In [ ]:
readmit_by_diag = df.groupby('diag_1_group')['readmit_30'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(readmit_by_diag.index, readmit_by_diag.values, color='teal', edgecolor='white')
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('30-Day Readmission Rate by Primary Diagnosis Group')
for i, val in enumerate(readmit_by_diag.values):
    ax.text(i, val + 0.1, f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 3-2. A1Cresult vs. Insulin Use

In [ ]:
a1c_insulin = df.groupby('A1Cresult')['insulin_binary'].mean() * 100
a1c_order_labels = ['None', 'Norm', '>7', '>8']
a1c_insulin = a1c_insulin.reindex([l for l in a1c_order_labels if l in a1c_insulin.index])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(a1c_insulin.index, a1c_insulin.values, color='mediumpurple', edgecolor='white')
ax.set_xlabel('A1C Result')
ax.set_ylabel('Insulin Use Rate (%)')
ax.set_title('Insulin Use Rate by A1C Test Result')
for bar, val in zip(bars, a1c_insulin.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 3-3. Correlation Heatmap — Targets vs Admission-Time Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

heatmap_df = df[['insulin_binary', 'time_in_hospital']].copy()
heatmap_df['A1Cresult_enc']    = df['A1Cresult'].map({'None': 0, 'Norm': 1, '>7': 2, '>8': 3})
heatmap_df['number_outpatient'] = df['number_outpatient']
heatmap_df['number_emergency']  = df['number_emergency']
heatmap_df['number_inpatient']  = df['number_inpatient']

le = LabelEncoder()
heatmap_df['gender_enc']        = le.fit_transform(df['gender'])
heatmap_df['race_enc']          = le.fit_transform(df['race'])
heatmap_df['age_group_enc']     = df['age_group'].map({'Young': 0, 'Middle': 1, 'Older': 2})
heatmap_df['admission_type_enc']= le.fit_transform(df['admission_type'])
heatmap_df['diag_1_group_enc']  = le.fit_transform(df['diag_1_group'])

target_cols  = ['insulin_binary', 'time_in_hospital']
feature_cols = ['gender_enc', 'race_enc', 'age_group_enc', 'admission_type_enc',
                'diag_1_group_enc', 'A1Cresult_enc',
                'number_outpatient', 'number_emergency', 'number_inpatient']

corr_data = heatmap_df[target_cols + feature_cols].corr().loc[target_cols, feature_cols]

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 11}, ax=ax)
ax.set_title('Correlation — Targets vs Admission-Time Features', pad=12)
ax.set_ylabel('Target')
plt.tight_layout()
plt.show()

## 3-4. Insulin Use Rate by Age Group

In [ ]:
insulin_by_age = df.groupby('age_group')['insulin_binary'].mean() * 100
age_order = ['Young', 'Middle', 'Older']
insulin_by_age = insulin_by_age.reindex(age_order)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(insulin_by_age.index, insulin_by_age.values,
              color=['#4e9af1', '#f1a34e', '#e05c5c'], edgecolor='white', width=0.5)
ax.set_xlabel('Age Group')
ax.set_ylabel('Insulin Use Rate (%)')
ax.set_title('Insulin Use Rate by Age Group')
ax.set_ylim(0, 80)
for bar, val in zip(bars, insulin_by_age.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 3-5. Prior Inpatient Visits vs Time in Hospital

In [ ]:
df['inpatient_group'] = pd.cut(
    df['number_inpatient'],
    bins=[-1, 0, 1, 2, 3, 100],
    labels=['0', '1', '2', '3', '4+']
)

fig, ax = plt.subplots(figsize=(9, 5))
groups = [df[df['inpatient_group'] == g]['time_in_hospital'].dropna()
          for g in ['0', '1', '2', '3', '4+']]
bp = ax.boxplot(groups, labels=['0', '1', '2', '3', '4+'],
                patch_artist=True, notch=False,
                boxprops=dict(facecolor='lightsteelblue'),
                medianprops=dict(color='navy', linewidth=2))
ax.set_xlabel('Prior Year Inpatient Visits')
ax.set_ylabel('Time in Hospital (days)')
ax.set_title('Prior Inpatient Visits vs Length of Stay')
means = [g.mean() for g in groups]
ax.plot(range(1, 6), means, 'D--', color='salmon', markersize=6, label='Mean')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Encoding

In [ ]:
# Ordinal encoding
a1c_order = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
df['A1Cresult_enc'] = df['A1Cresult'].map(a1c_order)

# Binary encoding
df['gender_enc'] = (df['gender'] == 'Male').astype(int)

# One-hot encoding for nominal categoricals
# Including all 3 diagnoses: primary (diag_1) plus secondary (diag_2, diag_3).
# The model learns separate coefficients for each, so it naturally weighs
# primary diagnosis higher than secondary ones based on the data.
# Patients with no secondary diagnosis fall into 'Other' (baseline) -> all-zero rows.
ohe_cols = ['race', 'age_group', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'admission_type']
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

# Convert boolean OHE columns to int (so select_dtypes keeps them)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# Memory optimization: downcast OHE columns to int8 (values are only 0 or 1)
ohe_int_cols = [c for c in df.columns if any(
    c.startswith(p) for p in ['race_', 'age_group_', 'diag_1_group_', 'diag_2_group_', 'diag_3_group_', 'admission_type_']
)]
for c in ohe_int_cols:
    df[c] = df[c].astype('int8')

print('Shape after encoding:', df.shape)
print(f'Boolean columns converted to int: {len(bool_cols)}')
print(f'OHE columns downcast to int8: {len(ohe_int_cols)}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

# 4. Modeling

### Feature Set Design Rationale

Both classification and regression use only information available **at or before admission**. In-stay features (`num_lab_procedures`, `num_medications`, `num_procedures`, `max_glu_serum`, `change`, `diabetesMed`) are excluded because they are determined **during** the stay, not knowable at admission time. Including them would mean predicting outcomes using their own consequences (data leakage).

`A1Cresult` IS kept: HbA1c reflects 2–3 month average glucose **prior to** admission.

### Models Used

- **Classification**: Logistic Regression and Decision Tree
- **Regression**: Linear Regression and Neural Network (MLP)
- **Clustering**: K-Means and DBSCAN

In [ ]:
ohe_generated = [c for c in df.columns if any(
    c.startswith(p) for p in ['race_', 'age_group_', 'diag_1_group_', 'diag_2_group_', 'diag_3_group_', 'admission_type_']
)]

admission_features = [
    'gender_enc',
    'A1Cresult_enc',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
] + ohe_generated

classification_features = admission_features
regression_features     = admission_features

print(f'Features used for both models ({len(admission_features)}):')
print(admission_features)

### Outlier Capping Function

As planned in Section 2-4, we define a capping function that will be applied **only to training data**. For IQR-applicable variables, we cap at 1.5×IQR bounds. For zero-inflated variables, we cap at the 99th percentile.

In [ ]:
def cap_outliers_train(X_train, iqr_cols, pctl_cols, pctl=0.99):
    """Cap outliers in training data only. Returns capped data and bounds dict."""
    bounds = {}
    X_capped = X_train.copy()

    # IQR-based capping
    for col in iqr_cols:
        if col not in X_capped.columns:
            continue
        Q1 = X_capped[col].quantile(0.25)
        Q3 = X_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        X_capped[col] = X_capped[col].clip(lo, hi)
        bounds[col] = (lo, hi)

    # Percentile-based capping (for zero-inflated distributions)
    for col in pctl_cols:
        if col not in X_capped.columns:
            continue
        hi = X_capped[col].quantile(pctl)
        lo = X_capped[col].quantile(1 - pctl)
        X_capped[col] = X_capped[col].clip(lo, hi)
        bounds[col] = (lo, hi)

    return X_capped, bounds

def apply_caps(X, bounds):
    """Apply pre-computed caps to test data."""
    X_capped = X.copy()
    for col, (lo, hi) in bounds.items():
        if col in X_capped.columns:
            X_capped[col] = X_capped[col].clip(lo, hi)
    return X_capped

# Define which columns get which treatment
iqr_cap_cols  = ['num_lab_procedures', 'num_procedures', 'num_medications']
pctl_cap_cols = ['number_outpatient', 'number_emergency', 'number_inpatient']

## 4-1. Classification: Predicting Insulin Use

**Task**: Predict whether insulin was used during a hospital encounter (binary: 0=No, 1=Yes)

**Models**: Logistic Regression and Decision Tree Classifier

In [ ]:
# Prepare data
X_cls = df[classification_features].copy().select_dtypes(include=[np.number])
y_cls = df['insulin_binary'].copy()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Apply outlier capping to training data ONLY
X_train_c, caps_c = cap_outliers_train(X_train_c, iqr_cap_cols, pctl_cap_cols)
X_test_c = apply_caps(X_test_c, caps_c)

print(f"Train: {X_train_c.shape}, Test: {X_test_c.shape}")
print(f"Caps applied (from training data): {caps_c}")

# Scale (use float32 to halve memory)
scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c).astype('float32')
X_test_c_sc  = scaler_c.transform(X_test_c).astype('float32')

import gc; gc.collect()

In [ ]:
# --- Model 1: Logistic Regression ---
lr_clf = LogisticRegression(max_iter=500, random_state=42)
lr_clf.fit(X_train_c_sc, y_train_c)
y_pred_lr = lr_clf.predict(X_test_c_sc)
y_prob_lr = lr_clf.predict_proba(X_test_c_sc)[:, 1]

print('=== Logistic Regression — Insulin Classification ===')
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_lr):.4f}')
print(f'ROC AUC:  {roc_auc_score(y_test_c, y_prob_lr):.4f}')
print()
print(classification_report(y_test_c, y_pred_lr, target_names=['No Insulin', 'Insulin Used']))

In [ ]:
# --- Model 2: Decision Tree ---
dt_clf = DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, random_state=42)
dt_clf.fit(X_train_c_sc, y_train_c)
y_pred_dt = dt_clf.predict(X_test_c_sc)
y_prob_dt = dt_clf.predict_proba(X_test_c_sc)[:, 1]

print('=== Decision Tree — Insulin Classification ===')
print(f'Accuracy: {accuracy_score(y_test_c, y_pred_dt):.4f}')
print(f'ROC AUC:  {roc_auc_score(y_test_c, y_prob_dt):.4f}')
print()
print(classification_report(y_test_c, y_pred_dt, target_names=['No Insulin', 'Insulin Used']))

In [ ]:
# Confusion Matrices — side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_lr, y_pred_dt],
                              ['Logistic Regression', 'Decision Tree']):
    cm = confusion_matrix(y_test_c, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Insulin', 'Insulin'], yticklabels=['No Insulin', 'Insulin'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {title}')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves — both models
fig, ax = plt.subplots(figsize=(8, 6))

for y_prob, label in [(y_prob_lr, 'Logistic Regression'), (y_prob_dt, 'Decision Tree')]:
    fpr, tpr, _ = roc_curve(y_test_c, y_prob)
    auc_val = roc_auc_score(y_test_c, y_prob)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Insulin Classification')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 features — Logistic Regression coefficients
coef_df = pd.DataFrame({
    'feature': X_cls.columns,
    'coefficient': lr_clf.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['salmon' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient')
ax.set_title('Top 10 Features — Logistic Regression (Insulin)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Free classification memory before next section
del lr_clf, dt_clf, X_train_c, X_test_c, X_train_c_sc, X_test_c_sc, X_cls
import gc; gc.collect()

## 4-2. Regression: Predicting Length of Stay

**Task**: Predict `time_in_hospital` (continuous, 1–14 days)

**Models**: Linear Regression and Neural Network (MLP) Regressor

**Why these two?** During experimentation we tested Linear Regression, Ridge, ElasticNet (with CV-tuned alpha), Decision Tree Regressor, and a Neural Network. Linear, Ridge, and ElasticNet all produced essentially identical R² scores (~0.092), suggesting multicollinearity was not an issue and regularization had no effect. Decision Tree was slightly worse (R² ~0.079). Only the Neural Network achieved a meaningfully better R² (~0.107) by capturing non-linear interactions among features. We therefore present Linear Regression as a clean baseline and the MLP as a non-linear improvement.

In [ ]:
X_reg = df[regression_features].copy().select_dtypes(include=[np.number])
y_reg = df['time_in_hospital'].copy()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Apply outlier capping to training data ONLY
X_train_r, caps_r = cap_outliers_train(X_train_r, iqr_cap_cols, pctl_cap_cols)
X_test_r = apply_caps(X_test_r, caps_r)

# Scale (float32 for memory)
scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r).astype('float32')
X_test_r_sc  = scaler_r.transform(X_test_r).astype('float32')

print(f"Train: {X_train_r.shape}, Test: {X_test_r.shape}")

# Free intermediate
del X_reg
import gc; gc.collect()

In [ ]:
# --- Model 1: Linear Regression ---
lr_reg = LinearRegression()
lr_reg.fit(X_train_r_sc, y_train_r)
y_pred_lr_r = lr_reg.predict(X_test_r_sc)

rmse_lr = np.sqrt(mean_squared_error(y_test_r, y_pred_lr_r))
mae_lr  = mean_absolute_error(y_test_r, y_pred_lr_r)
r2_lr   = r2_score(y_test_r, y_pred_lr_r)

print('=== Linear Regression — Time in Hospital ===')
print(f'RMSE: {rmse_lr:.4f} days')
print(f'MAE:  {mae_lr:.4f} days')
print(f'R²:   {r2_lr:.4f}')
print(f'Baseline (predict mean): RMSE = {np.std(y_test_r):.4f}')

In [ ]:
# --- Model 2: Neural Network (MLP) Regressor ---
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    learning_rate='adaptive',
    alpha=0.001
)
mlp_reg.fit(X_train_r_sc, y_train_r)
y_pred_mlp = mlp_reg.predict(X_test_r_sc)

rmse_mlp = np.sqrt(mean_squared_error(y_test_r, y_pred_mlp))
mae_mlp  = mean_absolute_error(y_test_r, y_pred_mlp)
r2_mlp   = r2_score(y_test_r, y_pred_mlp)

print('=== Neural Network (MLP) — Time in Hospital ===')
print(f'RMSE:    {rmse_mlp:.4f} days')
print(f'MAE:     {mae_mlp:.4f} days')
print(f'R²:      {r2_mlp:.4f}')
print(f'Epochs:  {mlp_reg.n_iter_} (stopped early)')

In [ ]:
# Regression comparison table
print('\n=== Regression Model Comparison ===')
print(f'{"Model":<28} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('-' * 55)
print(f'{"Linear Regression":<28} {rmse_lr:>8.4f} {mae_lr:>8.4f} {r2_lr:>8.4f}')
print(f'{"Neural Network (MLP)":<28} {rmse_mlp:>8.4f} {mae_mlp:>8.4f} {r2_mlp:>8.4f}')
print('-' * 55)
print(f'{"Baseline (predict mean)":<28} {np.std(y_test_r):>8.4f} {"":>8} {"0.0000":>8}')

In [ ]:
# Visualization: Actual vs Predicted + Residuals for both models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (y_pred, title) in enumerate([(y_pred_lr_r, 'Linear Regression'),
                                      (y_pred_mlp,  'Neural Network (MLP)')]):
    # Actual vs Predicted
    axes[0, i].scatter(y_test_r, y_pred, alpha=0.1, color='steelblue', s=5)
    axes[0, i].plot([1, 14], [1, 14], 'r--', linewidth=1.5)
    axes[0, i].set_xlabel('Actual Days')
    axes[0, i].set_ylabel('Predicted Days')
    axes[0, i].set_title(f'Actual vs Predicted — {title}')

    # Residual distribution
    residuals = y_test_r - y_pred
    axes[1, i].hist(residuals, bins=40, color='salmon', edgecolor='white')
    axes[1, i].axvline(0, color='black', linewidth=1)
    axes[1, i].set_xlabel('Residual (Actual − Predicted)')
    axes[1, i].set_ylabel('Count')
    axes[1, i].set_title(f'Residual Distribution — {title}')

plt.tight_layout()
plt.show()

# Free regression model memory before clustering
del lr_reg, mlp_reg, X_train_r, X_test_r, X_train_r_sc, X_test_r_sc
import gc; gc.collect()

## 4-3. Clustering: Discovering Patient Subgroups

**Task**: Identify natural patient subgroups using clinical and demographic features

**Models**: K-Means and DBSCAN

In [ ]:
# Clustering features — use OHE'd diagnosis groups (1, 2, 3), age group,
# and numeric clinical features.
cluster_features = (
    [c for c in df.columns if c.startswith('diag_1_group_')] +
    [c for c in df.columns if c.startswith('diag_2_group_')] +
    [c for c in df.columns if c.startswith('age_group_')] +
    ['num_medications', 'num_lab_procedures', 'number_diagnoses', 'A1Cresult_enc']
)
cluster_features = [f for f in cluster_features if f in df.columns]

X_clust = df[cluster_features].copy().select_dtypes(include=[np.number]).dropna()

# Memory optimization: convert to float32 (default float64 doubles memory usage)
X_clust = X_clust.astype('float32')

scaler_k = StandardScaler()
X_clust_sc = scaler_k.fit_transform(X_clust).astype('float32')

print(f"Clustering data shape: {X_clust.shape}")
print(f"Features ({len(cluster_features)}): {cluster_features}")
print(f"Memory: {X_clust_sc.nbytes / 1024**2:.1f} MB")

In [ ]:
# Elbow method + Silhouette analysis
# Use MiniBatchKMeans for k exploration (faster, much lower memory than full KMeans)
from sklearn.cluster import MiniBatchKMeans

inertias = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=2048)
    labels = km.fit_predict(X_clust_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust_sc, labels, sample_size=2000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method — K-Means')

axes[1].plot(list(k_range), sil_scores, 's-', color='salmon')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')

plt.tight_layout()
plt.show()

best_k_sil = list(k_range)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k_sil} (score={max(sil_scores):.4f})")

# Trade-off: silhouette may favor very high k or k=2 (too coarse).
# We choose k that balances silhouette with clinical interpretability.
# For clinical profiling, 4-6 clusters is typically the sweet spot.
if best_k_sil <= 2:
    best_k = 4
elif best_k_sil > 6:
    # Find best k within 4-6 range
    sil_in_range = [(k, s) for k, s in zip(k_range, sil_scores) if 4 <= k <= 6]
    best_k = max(sil_in_range, key=lambda x: x[1])[0]
else:
    best_k = best_k_sil
print(f"Selected k={best_k} for modeling (balancing silhouette with interpretability)")

In [ ]:
# --- Model 1: K-Means (using MiniBatchKMeans for memory efficiency) ---
kmeans = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=10, batch_size=2048)
km_labels = kmeans.fit_predict(X_clust_sc)
df.loc[X_clust.index, 'cluster_km'] = km_labels

km_sil = silhouette_score(X_clust_sc, km_labels, sample_size=2000, random_state=42)
print(f'K-Means (k={best_k})')
print(f'  Inertia:          {kmeans.inertia_:.0f}')
print(f'  Silhouette Score: {km_sil:.4f}')
print(f'  Cluster sizes:')
print(pd.Series(km_labels).value_counts().sort_index())

In [ ]:
# --- Model 2: DBSCAN ---
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# DBSCAN is O(n^2) memory in worst case, so we subsample more aggressively
np.random.seed(42)
sample_idx = np.random.choice(len(X_clust_sc), size=min(10000, len(X_clust_sc)), replace=False)
X_db_sample = X_clust_sc[sample_idx].astype('float32')

# k-distance plot to choose eps
nn = NearestNeighbors(n_neighbors=50)
nn.fit(X_db_sample)
distances, _ = nn.kneighbors(X_db_sample)
k_dist = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_dist, color='steelblue')
ax.set_xlabel('Points (sorted)')
ax.set_ylabel('50-NN Distance')
ax.set_title('k-Distance Plot for DBSCAN eps Selection')
ax.axhline(y=4.0, color='red', linestyle='--', label='eps=4.0')
ax.legend()
plt.tight_layout()
plt.show()

# Fit DBSCAN with chosen eps
dbscan = DBSCAN(eps=4.0, min_samples=50)
db_labels_sample = dbscan.fit_predict(X_db_sample)

n_clusters_db = len(set(db_labels_sample)) - (1 if -1 in db_labels_sample else 0)
n_noise = (db_labels_sample == -1).sum()

print(f'DBSCAN Results (on {len(X_db_sample)} samples):')
print(f'  Clusters found: {n_clusters_db}')
print(f'  Noise points:   {n_noise} ({n_noise/len(db_labels_sample)*100:.1f}%)')
print(f'  Cluster sizes:')
print(pd.Series(db_labels_sample).value_counts().sort_index())

# Silhouette for DBSCAN (excluding noise)
db_sil = None
if n_clusters_db >= 2:
    mask = db_labels_sample != -1
    if mask.sum() > 100:
        db_sil = silhouette_score(X_db_sample[mask], db_labels_sample[mask], sample_size=2000, random_state=42)
        print(f'  Silhouette Score (excl. noise): {db_sil:.4f}')

# Free memory after DBSCAN
del nn, distances, k_dist
import gc; gc.collect()

In [ ]:
# PCA Visualization — K-Means vs DBSCAN
pca_vis = PCA(n_components=2, random_state=42)
X_pca_2d = pca_vis.fit_transform(X_clust_sc)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# K-Means (full data)
scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                            c=km_labels, cmap='tab10', alpha=0.4, s=8)
axes[0].set_title(f'K-Means (k={best_k})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# DBSCAN (subsampled)
X_pca_sample = X_pca_2d[sample_idx]
scatter2 = axes[1].scatter(X_pca_sample[:, 0], X_pca_sample[:, 1],
                            c=db_labels_sample, cmap='tab10', alpha=0.4, s=8)
axes[1].set_title(f'DBSCAN (clusters={n_clusters_db}, noise={n_noise})')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.suptitle('Cluster Visualization via PCA (2D)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"PCA explained variance: PC1={pca_vis.explained_variance_ratio_[0]:.2%}, PC2={pca_vis.explained_variance_ratio_[1]:.2%}")

In [ ]:
# K-Means cluster profiling — detailed
cluster_summary = df.groupby('cluster_km').agg(
    n_patients        = ('insulin_binary', 'count'),
    avg_time_hospital = ('time_in_hospital', 'mean'),
    insulin_use_rate  = ('insulin_binary', 'mean'),
    readmit_30_rate   = ('readmit_30', 'mean'),
    avg_medications   = ('num_medications', 'mean'),
    avg_diagnoses     = ('number_diagnoses', 'mean')
).round(3)

cluster_summary['insulin_use_rate'] = (cluster_summary['insulin_use_rate'] * 100).round(1).astype(str) + '%'
cluster_summary['readmit_30_rate']  = (cluster_summary['readmit_30_rate']  * 100).round(1).astype(str) + '%'

print('=== K-Means Cluster Profiling ===')
print(cluster_summary.to_string())

# Reconstruct age group from OHE columns (drop_first=True dropped 'Middle')
def get_age(row):
    if row.get('age_group_Older', 0) == 1: return 'Older'
    if row.get('age_group_Young', 0) == 1: return 'Young'
    return 'Middle'
df['_age_group_str'] = df.apply(get_age, axis=1)

# Reconstruct primary diagnosis group from OHE columns
diag_cols = [c for c in df.columns if c.startswith('diag_1_group_')]
def get_diag1(row):
    for c in diag_cols:
        if row.get(c, 0) == 1:
            return c.replace('diag_1_group_', '')
    return 'Other'  # baseline
df['_diag_1_str'] = df.apply(get_diag1, axis=1)

# Reconstruct A1Cresult tested status
df['_a1c_tested'] = (df['A1Cresult_enc'] != 0).astype(int)

print('\n=== Per-Cluster Clinical Profile ===')
for cl in sorted(df['cluster_km'].dropna().unique()):
    cl = int(cl)
    sub = df[df['cluster_km'] == cl]
    n = len(sub)
    pct = n / df['cluster_km'].notna().sum() * 100

    age_dist = sub['_age_group_str'].value_counts(normalize=True) * 100
    top_diag = sub['_diag_1_str'].value_counts(normalize=True).head(2) * 100
    a1c_tested = sub['_a1c_tested'].mean() * 100
    male_pct = sub['gender_enc'].mean() * 100

    print(f'\nCluster {cl}: {n} patients ({pct:.1f}%)')
    print(f'  Age:        Young={age_dist.get("Young", 0):.0f}%, Middle={age_dist.get("Middle", 0):.0f}%, Older={age_dist.get("Older", 0):.0f}%')
    print(f'  Male:       {male_pct:.0f}%')
    diag_str = ", ".join([f"{d}={v:.0f}%" for d, v in top_diag.items()])
    print(f'  Top diag:   {diag_str}')
    print(f'  A1C tested: {a1c_tested:.0f}%')
    print(f'  Insulin:    {sub["insulin_binary"].mean()*100:.0f}%, Readmit30: {sub["readmit_30"].mean()*100:.0f}%')

# Clean up helpers
df = df.drop(columns=['_age_group_str', '_diag_1_str', '_a1c_tested'])

In [ ]:
# Cluster comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cg = df.groupby('cluster_km')

cg['time_in_hospital'].mean().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Time in Hospital by Cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Days')

(cg['insulin_binary'].mean() * 100).plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Insulin Use Rate by Cluster (%)')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('%')

(cg['readmit_30'].mean() * 100).plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='white')
axes[2].set_title('30-Day Readmission Rate by Cluster (%)')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('%')

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# 5. Model Comparison and Summary

In [ ]:
# Classification summary
print('=' * 65)
print('CLASSIFICATION — Insulin Use Prediction')
print('=' * 65)
print(f'{"Model":<25} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>8} {"AUC":>8}')
print('-' * 75)

for name, yp, yprob in [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    ('Decision Tree',       y_pred_dt, y_prob_dt),
]:
    acc = accuracy_score(y_test_c, yp)
    prec = precision_score(y_test_c, yp)
    rec = recall_score(y_test_c, yp)
    f1 = f1_score(y_test_c, yp)
    auc = roc_auc_score(y_test_c, yprob)
    print(f'{name:<25} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>8.4f} {auc:>8.4f}')

print()
print('=' * 65)
print('REGRESSION — Time in Hospital Prediction')
print('=' * 65)
print(f'{"Model":<25} {"RMSE":>10} {"MAE":>10} {"R²":>10}')
print('-' * 60)
print(f'{"Linear Regression":<25} {rmse_lr:>10.4f} {mae_lr:>10.4f} {r2_lr:>10.4f}')
print(f'{"Neural Network (MLP)":<25} {rmse_mlp:>10.4f} {mae_mlp:>10.4f} {r2_mlp:>10.4f}')

print()
print('=' * 65)
print('CLUSTERING — Patient Subgroups')
print('=' * 65)
print(f'{"Model":<25} {"Clusters":>10} {"Silhouette":>12}')
print('-' * 60)
print(f'{"K-Means":<25} {best_k:>10} {km_sil:>12.4f}')
if db_sil is not None:
    print(f'{"DBSCAN":<25} {n_clusters_db:>10} {db_sil:>12.4f}')
else:
    print(f'{"DBSCAN":<25} {n_clusters_db:>10} {"N/A":>12}')

# 6. Key Insights and Reflections

### Main Findings

**Classification (insulin use)**: Both models achieve roughly 60% accuracy with ROC AUC ~0.64 using only admission-time features. The Decision Tree provides slightly better recall (catching more insulin users), while Logistic Regression has higher precision. The strongest predictors are A1Cresult (higher A1C → more insulin), primary diagnosis being diabetes, and prior inpatient visit count.

**Regression (length of stay)**: Linear Regression achieves R² ≈ 0.092, and the Neural Network improves this to R² ≈ 0.107 by capturing non-linear interactions. Both substantially outperform the mean baseline (R²=0). The modest R² reflects a real finding rather than a modeling failure: hospital duration is largely determined by in-stay events (complications, response to treatment), which we deliberately exclude to prevent data leakage.

**Clustering (patient subgroups)**: K-Means with k=6 identified clinically meaningful subgroups. Naming each by its dominant characteristics:

1. **Young Diabetic Group** (~3%): 100% young patients, 50% with primary diabetes diagnosis. Highest insulin use rate (75%) but **lowest readmission rate (6%)**, shortest hospital stay (3.2 days), and fewest medications. Suggests young diabetic patients are managed actively but recover quickly.

2. **High-Acuity Polypharmacy Group** (~14%): Highest A1C testing rate (79%), most medications (avg 19), longest stay (5.4 days), and 68% insulin use. Mixed-age, slightly male-skewed. This is the most clinically intensive group — likely poorly-controlled diabetic patients with multiple comorbidities.

3. **Respiratory Group** (~13%): 100% respiratory primary diagnosis, predominantly older (68%). Average insulin use, low readmission. Diabetes is a comorbidity here, not the primary reason for admission.

4. **General Elderly Cohort** (~60%): The largest cluster — predominantly older patients with non-major-category diagnoses ('Other' 64%, Digestive 11%). Low A1C testing (4%) suggests diabetes is being managed but not actively re-evaluated during this admission.

5. **Injury Group** (~6%): 100% injury primary diagnosis, predominantly older (72%) and slightly female-skewed. **Highest 30-day readmission rate (11%)** — likely reflects recurring fall-related injuries among elderly patients.

6. **Digestive Group** (~4%): Mixed diagnoses with digestive disease prominent (43%). Moderate insulin use, lower readmission.

**Key clinical takeaway**: Two clusters stand out as actionable:
- The **High-Acuity Polypharmacy Group** identifies patients most needing intensive endocrinology resources at admission.
- The **Injury Group** identifies elderly patients most likely to be readmitted, suggesting opportunities for post-discharge fall-prevention programs.

### Surprising Findings

The most notable was the OHE boolean bug: `pd.get_dummies()` creates boolean-typed columns that `select_dtypes(include=[np.number])` silently drops. In the interim version this excluded 22 of 28 features from modeling without any error. Fixing it (converting boolean to int) immediately raised classification accuracy from ~55% to ~60% and regression R² from ~0.06 to ~0.09. A reminder that quiet preprocessing failures can be more dangerous than loud ones.

Another surprise was that log-transforming the regression target (a textbook fix for right-skew) actually *hurt* R² in our case. The transformation made the residuals more Gaussian, but the inverse transform amplified errors on long stays, where R² is most sensitive. We left the target on the original scale and noted the experiment.

### Limitations and Ethical Considerations

- **Feature restriction limits accuracy**: Excluding in-stay features is methodologically necessary but caps predictive power. A clinical deployment would need real-time signals.
- **Demographic skew**: The dataset is over 75% Caucasian. Models may not generalize equitably across populations.
- **Temporal gap**: Data covers 1999–2008. Modern diabetes care has changed considerably.
- **Not for clinical use**: These models illustrate patterns; they should not drive treatment decisions without rigorous validation.

### What I Would Do Differently

- Try ensemble methods (Random Forest, Gradient Boosting / XGBoost) — they typically beat single models on structured healthcare data.
- Add interaction features (e.g., A1C level × diagnosis group) to help linear models capture combined effects.
- Reconsider including `medical_specialty` with a careful leakage analysis — Strack et al. (2014) kept it; we excluded it conservatively but the trade-off is worth re-examining.